# Red Cedar — Training Settings

This notebook shows training settings you can change, one section per setting. Each section trains a small model on the same dataset (4 qubits, 1 layer) and tests it, so the effect of the setting is easy to see.

Run the setup cell first; after that, each section can be run on its own. For the basic workflow see `01_quick_start.ipynb`. All settings are listed in the settings table of the redcedar README.

In [ ]:
import redcedar

# Connect to the server
# Reads REDCEDAR_URL and REDCEDAR_API_KEY from environment, or pass explicitly:
# client = redcedar.connect(url="http://localhost:8002", api_key="your-key")
client = redcedar.connect()
print(f"Red Cedar SDK v{redcedar.__version__}")
print(f"Server: {client.url}")
print(f"Status: {client.status()}")

dataset = "wdbc"

# Small, quick settings shared by every section; each section adds one setting.
base = {"num_qubits": 4, "min_layers": 1, "max_layers": 1}

def train_and_test(settings):
    """Train with these settings, test on the held-out data, return (train_id, accuracy, loss)."""
    result = client.training.run(dataset, settings=settings)
    test = client.training.evaluate(result["train_id"])
    return result["train_id"], test["overall_accuracy"], test["overall_loss"]

## 1. `seed`: repeatable runs

Training starts from random parameters, so two runs with the same settings usually end differently. `seed` fixes the starting parameters (and, on a CUDA-Q server, the circuit sampling), so a run can be repeated exactly. Leave it out, or set it to `None`, for a random start.

On a model this small the test accuracy often comes out the same; the loss shows the difference between runs.

In [ ]:
runs = {
    "seed 123, run 1": dict(base, seed=123),
    "seed 123, run 2": dict(base, seed=123),
    "no seed":         dict(base),
}
results = {name: train_and_test(settings) for name, settings in runs.items()}
for name, (train_id, accuracy, loss) in results.items():
    print(f"{name:16s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

same = results["seed 123, run 1"][1:] == results["seed 123, run 2"][1:]
print(f"\nSame seed, same result: {same}")

## 2. `op_direction`: order of the operators

Each layer pairs the prediction qubit with every data qubit in turn. `op_direction` sets the order: `"backward"` (the default) starts at the highest data qubit, `"forward"` at the lowest. The number of parameters is the same either way; the results differ because the optimizer visits the parameters in a different order.

Both runs below use the same seed, so the direction is the only difference. Neither direction is better in general: across many seeds and datasets the results were close.

In [ ]:
for direction in ["backward", "forward"]:
    train_id, accuracy, loss = train_and_test(dict(base, seed=123, op_direction=direction))
    print(f"{direction:8s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

## Settings are saved with the model

Each trained model keeps its settings, and the model list shows them, so any run can be traced back and repeated.

In [ ]:
models = client.inference.models()["models"]
for m in models[:5]:
    p = m["training_params"]
    print(f"{m['train_id'][:8]}  {m['dataset']:12s} seed={p['seed']}  op_direction={p['op_direction']}")